In [14]:
import pandas as pd
import numpy as np

dfc = pd.read_csv("cust_profile.csv")
dft = pd.read_csv("transactions.csv")

dfc.head()

,CustomerID,Name,City,JoinDate
0,C01,Ali,Karachi,2021-01-10
1,C02,Sara,Lahore,2020-03-15
2,C03,Usman,Islamabad,2019-07-22
3,C04,Ayesha,Karachi,2022-05-10
4,C05,Hassan,Lahore,2021-11-01


In [15]:
df = pd.merge(dfc,dft, on="CustomerID", how="left")
df

,CustomerID,Name,City,JoinDate,TransactionID,Date,Amount
0,C01,Ali,Karachi,2021-01-10,T001,7/1/2023,5000
1,C02,Sara,Lahore,2020-03-15,T002,6/20/2023,20000
2,C03,Usman,Islamabad,2019-07-22,T003,6/15/2023,7000
3,C04,Ayesha,Karachi,2022-05-10,T004,12/15/2022,8000
4,C05,Hassan,Lahore,2021-11-01,T005,5/10/2023,15000
...,...,...,...,...,...,...,...
205,C206,Hamza,Karachi,2020-05-10,T206,11/12/2022,7100
206,C207,Bilal,Lahore,2023-11-05,T207,7/28/2021,10800
207,C208,Mahnoor,Islamabad,2018-08-15,T208,5/20/2022,15500
208,C209,Yusuf,Hyderabad,2022-06-18,T209,10/18/2022,8000


In [16]:
df["Date"] = pd.to_datetime(df["Date"])
df["JoinDate"] = pd.to_datetime(df["JoinDate"])
df

,CustomerID,Name,City,JoinDate,TransactionID,Date,Amount
0,C01,Ali,Karachi,2021-01-10,T001,2023-07-01,5000
1,C02,Sara,Lahore,2020-03-15,T002,2023-06-20,20000
2,C03,Usman,Islamabad,2019-07-22,T003,2023-06-15,7000
3,C04,Ayesha,Karachi,2022-05-10,T004,2022-12-15,8000
4,C05,Hassan,Lahore,2021-11-01,T005,2023-05-10,15000
...,...,...,...,...,...,...,...
205,C206,Hamza,Karachi,2020-05-10,T206,2022-11-12,7100
206,C207,Bilal,Lahore,2023-11-05,T207,2021-07-28,10800
207,C208,Mahnoor,Islamabad,2018-08-15,T208,2022-05-20,15500
208,C209,Yusuf,Hyderabad,2022-06-18,T209,2022-10-18,8000


In [17]:
df.isnull().sum()

CustomerID       0
Name             0
City             0
JoinDate         0
TransactionID    0
Date             0
Amount           0
dtype: int64

In [18]:
today = pd.to_datetime("2023-07-20")

In [19]:
recency = df.groupby("CustomerID")["Date"].max().reset_index()
recency['Recency'] = (today - recency["Date"]).dt.days
recency

,CustomerID,Date,Recency
0,C01,2023-07-01,19
1,C02,2023-06-20,30
2,C03,2023-06-15,35
3,C04,2022-12-15,217
4,C05,2023-05-10,71
...,...,...,...
205,C95,2022-10-22,271
206,C96,2023-03-05,137
207,C97,2022-02-28,507
208,C98,2021-04-10,831


In [20]:
frequency = df.groupby("CustomerID")["TransactionID"].count().reset_index()
frequency.columns = ["CustomerID","Frequency"]
frequency

,CustomerID,Frequency
0,C01,1
1,C02,1
2,C03,1
3,C04,1
4,C05,1
...,...,...
205,C95,1
206,C96,1
207,C97,1
208,C98,1


In [21]:
monetary = df.groupby("CustomerID")["Amount"].sum().reset_index()
monetary.columns = ["CustomerID", "Monetary"]
monetary

,CustomerID,Monetary
0,C01,5000
1,C02,20000
2,C03,7000
3,C04,8000
4,C05,15000
...,...,...
205,C95,19800
206,C96,7700
207,C97,11200
208,C98,12800


In [22]:
rfm = recency.merge(frequency, on="CustomerID").merge(monetary, on="CustomerID")
rfm.head()
rfm = rfm[rfm["Frequency"] > 0]

In [23]:
rfm["R_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    q=5,
    labels=[1,2,3,4,5]
)

rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    5,
    labels=[1,2,3,4,5]
)
rfm["M_Score"] = pd.qcut(
    rfm["Monetary"].rank(method="first"),
    5,
    labels=[1,2,3,4,5]
)

In [24]:
rfm["RFM_Score"] = rfm["R_Score"].astype(int) + rfm["F_Score"].astype(int) + rfm["M_Score"].astype(int)
rfm
#👉 Higher score = best customers

,CustomerID,Date,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score
0,C01,2023-07-01,19,1,5000,1,1,1,3
1,C02,2023-06-20,30,1,20000,1,1,5,7
2,C03,2023-06-15,35,1,7000,1,1,2,4
3,C04,2022-12-15,217,1,8000,1,1,2,4
4,C05,2023-05-10,71,1,15000,1,1,4,6
...,...,...,...,...,...,...,...,...,...
205,C95,2022-10-22,271,1,19800,5,5,5,15
206,C96,2023-03-05,137,1,7700,5,5,2,12
207,C97,2022-02-28,507,1,11200,5,5,3,13
208,C98,2021-04-10,831,1,12800,5,5,4,14


In [25]:
rfm.shape

(210, 9)

In [36]:
rfm["Segment"] = np.where(rfm["RFM_Score"] >= 12, "VIP", np.where(rfm["RFM_Score"] >= 8, "Loyal", "At Risk"))
#VIP → top spenders
#Loyal → stable customers
#At Risk → need marketing

# EDA (Exploratory Data Analysis)

In [37]:
rfm["Segment"].value_counts() 

Loyal      84
At Risk    76
VIP        50
Name: Segment, dtype: int64

In [38]:
rfm.sort_values(by="RFM_Score", ascending=False)

,CustomerID,Date,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
185,C75,2022-08-08,346,1,20500,5,5,5,15,VIP
183,C73,2022-07-12,373,1,39500,5,5,5,15,VIP
205,C95,2022-10-22,271,1,19800,5,5,5,15,VIP
203,C93,2023-03-18,124,1,43500,5,5,5,15,VIP
193,C83,2021-03-20,852,1,36500,5,5,5,15,VIP
...,...,...,...,...,...,...,...,...,...,...
22,C111,2023-05-12,69,1,5900,1,1,1,3,At Risk
5,C06,2023-07-10,10,1,6000,1,1,1,3,At Risk
32,C120,2021-03-30,842,1,4400,1,1,1,3,At Risk
31,C12,2023-07-12,8,1,3000,1,1,1,3,At Risk


In [40]:
rfm.groupby("Segment")["Monetary"].mean().sort_values(ascending=False)

Segment
VIP        17404.000000
Loyal      13777.380952
At Risk    10652.631579
Name: Monetary, dtype: float64